Data Loading

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Check Spark version and test execution
print("Spark version:", spark.version)
spark.range(5).show()

Spark version: 4.2.0
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [0]:

from urllib.request import Request, urlopen

# Check the current workspace
spark.sql("""
    SELECT current_catalog() AS catalog,
           current_schema() AS schema
""").show()

# Test access to NYC TLC data
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"

try:
    req = Request(url, headers={"Range": "bytes=0-1023"})
    with urlopen(req, timeout=15) as response:
        response.read(16)
        print("Data access successful:", response.status)
except Exception as e:
    print("Direct access unavailable:", str(e)[:200])


+---------+-------+
|  catalog| schema|
+---------+-------+
|workspace|default|
+---------+-------+

Data access successful: 206


In [0]:

# Create a volume for NYC TLC data
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.nyc_tlc
""")

# Set the storage path
volume_path = "/Volumes/workspace/default/nyc_tlc"

# Verify the volume
print("Volume:", volume_path)
print(dbutils.fs.ls(volume_path))


Volume: /Volumes/workspace/default/nyc_tlc
[]


In [0]:

from urllib.request import urlopen
import shutil
import os

# Download January 2024 trip data
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"
file_path = "/Volumes/workspace/default/nyc_tlc/fhvhv_tripdata_2024-01.parquet"

if not os.path.exists(file_path):
    with urlopen(url, timeout=60) as response:
        with open(file_path, "wb") as file:
            shutil.copyfileobj(response, file, length=8 * 1024 * 1024)

# Check the downloaded file
size_mb = os.path.getsize(file_path) / (1024 ** 2)
print(f"Downloaded file size: {size_mb:.2f} MB")


Downloaded file size: 450.86 MB


In [0]:

# Load January data with PySpark
df = spark.read.parquet(file_path)

# Check schema and sample records
df.printSchema()
df.show(5)


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [0]:

import os
import shutil
from urllib.request import urlopen

# Download February and March data
months = ["2024-02", "2024-03"]

for month in months:
    filename = f"fhvhv_tripdata_{month}.parquet"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
    path = f"{volume_path}/{filename}"

    if not os.path.exists(path):
        temp_path = path + ".part"

        with urlopen(url, timeout=120) as response:
            with open(temp_path, "wb") as file:
                shutil.copyfileobj(response, file)

        os.replace(temp_path, path)

    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"{month}: {size_mb:.2f} MiB")

# Check total dataset size
all_months = ["2024-01", "2024-02", "2024-03"]

total_bytes = sum(
    os.path.getsize(
        f"{volume_path}/fhvhv_tripdata_{month}.parquet"
    )
    for month in all_months
)

print(f"Total dataset size: {total_bytes / (1024 ** 3):.2f} GiB")


2024-02: 441.05 MiB
2024-03: 484.41 MiB
Total dataset size: 1.34 GiB


In [0]:

# Load all three months
file_paths = [
    f"{volume_path}/fhvhv_tripdata_2024-{month}.parquet"
    for month in ["01", "02", "03"]
]

trips_raw = spark.read.parquet(*file_paths)

# Check dataset size
print("Total rows:", trips_raw.count())
print("Total columns:", len(trips_raw.columns))

# Preview selected columns
trips_raw.select(
    "pickup_datetime",
    "PULocationID",
    "trip_miles",
    "base_passenger_fare"
).show(5)


Total rows: 60303866
Total columns: 24
+-------------------+------------+----------+-------------------+
|    pickup_datetime|PULocationID|trip_miles|base_passenger_fare|
+-------------------+------------+----------+-------------------+
|2024-01-01 00:28:08|         161|      2.83|              45.61|
|2024-01-01 00:12:53|         137|      1.57|              10.05|
|2024-01-01 00:23:05|          79|      1.98|              18.07|
|2024-01-01 00:41:04|         234|      1.99|              17.17|
|2024-01-01 00:57:21|         148|      2.65|              38.67|
+-------------------+------------+----------+-------------------+
only showing top 5 rows


Data Cleaning & Transformations

In [0]:

from pyspark.sql import functions as F

# Filter invalid trips
trips_clean = (
    trips_raw
    .filter(F.col("trip_miles") > 0)
    .filter(F.col("base_passenger_fare") > 0)
    .select(
        "pickup_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "base_passenger_fare",
        "hvfhs_license_num"
    )
)

# Create new columns
trips_clean = (
    trips_clean
    .withColumn("pickup_month", F.month("pickup_datetime"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn(
        "fare_per_mile",
        F.round(
            F.col("base_passenger_fare") / F.col("trip_miles"), 2
        )
    )
)

# Preview cleaned data
trips_clean.show(5)

# Count valid trips
print("Valid trips:", trips_clean.count())


+-------------------+------------+------------+----------+-------------------+-----------------+------------+-----------+-------------+
|    pickup_datetime|PULocationID|DOLocationID|trip_miles|base_passenger_fare|hvfhs_license_num|pickup_month|pickup_hour|fare_per_mile|
+-------------------+------------+------------+----------+-------------------+-----------------+------------+-----------+-------------+
|2024-01-01 00:28:08|         161|         158|      2.83|              45.61|           HV0003|           1|          0|        16.12|
|2024-01-01 00:12:53|         137|          79|      1.57|              10.05|           HV0003|           1|          0|          6.4|
|2024-01-01 00:23:05|          79|         186|      1.98|              18.07|           HV0003|           1|          0|         9.13|
|2024-01-01 00:41:04|         234|         148|      1.99|              17.17|           HV0003|           1|          0|         8.63|
|2024-01-01 00:57:21|         148|          97| 

GroupBy & Aggregation

In [0]:

# Aggregate trips by month and hour
hourly_stats = (
    trips_clean
    .groupBy("pickup_month", "pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("base_passenger_fare"), 2).alias("avg_fare"),
        F.round(F.avg("trip_miles"), 2).alias("avg_miles")
    )
    .orderBy("pickup_month", "pickup_hour")
)

# Display aggregated results
hourly_stats.show(15)


+------------+-----------+----------+--------+---------+
|pickup_month|pickup_hour|trip_count|avg_fare|avg_miles|
+------------+-----------+----------+--------+---------+
|           1|          0|    696928|   23.36|     5.13|
|           1|          1|    492740|   23.94|     4.95|
|           1|          2|    368357|    24.4|     5.12|
|           1|          3|    296792|   25.07|     5.74|
|           1|          4|    295703|   27.55|     6.76|
|           1|          5|    344175|   28.07|     6.96|
|           1|          6|    548823|   25.96|     5.95|
|           1|          7|    863386|   24.28|     4.81|
|           1|          8|   1040550|   23.07|     4.37|
|           1|          9|    915622|   23.13|     4.55|
|           1|         10|    820211|    23.8|     4.76|
|           1|         11|    806574|   23.96|     4.73|
|           1|         12|    832265|   24.06|     4.74|
|           1|         13|    866961|   24.56|     4.82|
|           1|         14|    9

Spark SQL Query

In [0]:

# Register DataFrame as a SQL view
trips_clean.createOrReplaceTempView("trips")

# Query 1: Monthly trip summary
monthly_summary = spark.sql("""
    SELECT
        pickup_month,
        COUNT(*) AS total_trips,
        ROUND(AVG(base_passenger_fare), 2) AS avg_fare,
        ROUND(AVG(trip_miles), 2) AS avg_miles
    FROM trips
    GROUP BY pickup_month
    ORDER BY pickup_month
""")

monthly_summary.show()


+------------+-----------+--------+---------+
|pickup_month|total_trips|avg_fare|avg_miles|
+------------+-----------+--------+---------+
|           1|   19653311|   23.97|     4.84|
|           2|   19347478|   24.28|     4.93|
|           3|   21276651|   25.55|     4.95|
+------------+-----------+--------+---------+



In [0]:

# Query 2: Top 10 pickup locations
top_locations = spark.sql("""
    SELECT
        PULocationID,
        COUNT(*) AS total_trips,
        ROUND(AVG(base_passenger_fare), 2) AS avg_fare,
        ROUND(AVG(trip_miles), 2) AS avg_miles
    FROM trips
    WHERE PULocationID IS NOT NULL
    GROUP BY PULocationID
    ORDER BY total_trips DESC
    LIMIT 10
""")

top_locations.show()


+------------+-----------+--------+---------+
|PULocationID|total_trips|avg_fare|avg_miles|
+------------+-----------+--------+---------+
|         138|    1069589|   52.42|    11.52|
|         132|    1058813|   66.69|    18.06|
|          79|     842596|   24.28|     4.22|
|          61|     807515|   19.94|     3.69|
|         230|     738516|   35.33|     6.05|
|         161|     716223|   34.17|     5.23|
|         231|     702266|    29.5|     4.81|
|          68|     665960|   30.45|     5.13|
|         234|     656378|   27.61|     4.33|
|          37|     653577|   21.07|     3.97|
+------------+-----------+--------+---------+



Spark Execution Plan

In [0]:

# Inspect the physical execution plan
top_locations.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonTopK (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonTopK (7)
                     +- PhotonGroupingAgg (6)
                        +- PhotonShuffleExchangeSource (5)
                           +- PhotonShuffleMapStage (4)
                              +- PhotonShuffleExchangeSink (3)
                                 +- PhotonGroupingAgg (2)
                                    +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [3]: [PULocationID#10274, trip_miles#10276, base_passenger_fare#10278]
DictionaryFilters: [(trip_miles#10276 > 0.0), (base_passenger_fare#10278 > 0.0)]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/nyc_tlc/fhvhv_tripdata_2024-01.parquet, ... 2 entries]
ReadSchema: struct<PULocationID:int,t

Partitioning & Output Writing

In [0]:

# Set output path
output_path = f"{volume_path}/processed_trips"

# Write cleaned data partitioned by month
(
    trips_clean.write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(output_path)
)

# Check output files
display(dbutils.fs.ls(output_path))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_tlc/processed_trips/_SUCCESS,_SUCCESS,0,1795200750000
dbfs:/Volumes/workspace/default/nyc_tlc/processed_trips/pickup_month=1/,pickup_month=1/,0,1795200750656
dbfs:/Volumes/workspace/default/nyc_tlc/processed_trips/pickup_month=2/,pickup_month=2/,0,1795200750656
dbfs:/Volumes/workspace/default/nyc_tlc/processed_trips/pickup_month=3/,pickup_month=3/,0,1795200750656


In [0]:

# Load partitioned data
processed_df = spark.read.parquet(output_path)

# Filter March trips
march_trips = processed_df.filter(
    F.col("pickup_month") == 3
)

# Count March trips
print("March trips:", march_trips.count())

# Inspect partition pruning
march_trips.explain(mode="formatted")


March trips: 21276651
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [9]: [pickup_datetime#10759, PULocationID#10760, DOLocationID#10761, trip_miles#10762, base_passenger_fare#10763, hvfhs_license_num#10764, pickup_hour#10765, fare_per_mile#10766, pickup_month#10767]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/nyc_tlc/processed_trips]
PartitionFilters: [isnotnull(pickup_month#10767), (pickup_month#10767 = 3)]
ReadSchema: struct<pickup_datetime:timestamp_ntz,PULocationID:int,DOLocationID:int,trip_miles:double,base_passenger_fare:double,hvfhs_license_num:string,pickup_hour:int,fare_per_mile:double>

(2) PhotonColumnarToRow
Input [9]: [pickup_datetime#10759, PULocationID#10760, DOLocationID#10761, trip_miles#10762, base_passenger_fare#10763, hvfhs_license_num#10764, pickup_hour#10765, fare_per_mile#10766, pickup_month#10767]

(3) PhotonResultStage
Input [9]: [pickup_datetime#10759, PULoca

Actions vs Transformations

In [0]:

# Define transformations
lazy_demo = (
    processed_df
    .filter(F.col("pickup_month") == 1)
    .select("trip_miles", "base_passenger_fare")
    .withColumn(
        "fare_per_mile",
        F.col("base_passenger_fare") / F.col("trip_miles")
    )
)

print("Transformations defined without executing a Spark job.")

# Display the execution plan
lazy_demo.explain()

# Trigger execution with an action
print("Action: count()")
print("January trips:", lazy_demo.count())


Transformations defined without executing a Spark job.
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [trip_miles#10762, base_passenger_fare#10763, (base_passenger_fare#10763 / trip_miles#10762) AS fare_per_mile#10779]
      +- PhotonScan parquet [trip_miles#10762,base_passenger_fare#10763,pickup_month#10767] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/nyc_tlc/processed_trips], OptionalDataFilters: [], PartitionFilters: [isnotnull(pickup_month#10767), (pickup_month#10767 = 1)], ReadSchema: struct<trip_miles:double,base_passenger_fare:double>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
Action: count()
January trips: 19653311
